In [1]:
import sys
import os
import time
from lightkurve import search_targetpixelfile
from lightkurve import TessTargetPixelFile
import lightkurve as lk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from keras.models import load_model
from keras.optimizers import Adam
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Conv1D, MaxPooling1D, Flatten
from wotan import slide_clip
from wotan import transit_mask, flatten
from astropy.stats import sigma_clip
from astropy import units as u
import csv
import shutil
from scipy.interpolate import interp1d
from tsfresh import extract_features
from tsfresh.utilities.dataframe_functions import make_forecasting_frame
from tsfresh.utilities.dataframe_functions import impute
from statsmodels.tsa.seasonal import seasonal_decompose
from multiprocessing import Pool
import multiprocessing
import numpy as np
import pandas as pd
import lightkurve as lk
from scipy.signal import find_peaks
from astropy.timeseries import BoxLeastSquares
from scipy.interpolate import interp1d
from keras.models import Model
from keras.layers import Input, Dense, Conv1D, Flatten, Concatenate, Dropout
from keras.optimizers import Adam

/Users/nasvinnabeel/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
%matplotlib inline

Ensemble Model

In [3]:
value_df = 10000
cwd = os.getcwd()
dirname = cwd[len(cwd)-len("Satellite Datasets"):len(cwd)]
if(dirname != 'Satellite Datasets'):
    os.chdir('./Satellite Datasets')
star_check = pd.read_csv("updated_database_exoplanet.csv")
star_check = star_check.drop(['tid'],axis=1)
star_check_y = star_check[['confirmed_planet']]
star_check = star_check.reset_index().drop('index',axis=1)
star_check = star_check.apply(lambda row: row.fillna(0), axis=1)

In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(star_check.drop('confirmed_planet',axis=1),star_check[['confirmed_planet']], test_size=0.1, random_state=42)

In [5]:
X_train_flux = X_train.drop(['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx'],axis=1)
X_train_params = X_train[['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx']]
X_test_flux = X_test.drop(['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx'],axis=1)
X_test_params = X_test[['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx']]

In [6]:
input_flux = Input(shape=(10000,1))
x = Conv1D(filters=16, kernel_size=3, activation='relu')(input_flux)
x = MaxPooling1D(pool_size=2)(x)
x = Conv1D(filters=32, kernel_size=3, activation='relu')(x)
x = MaxPooling1D(pool_size=2)(x)
x = Flatten()(x)
model_flux = Model(inputs=input_flux, outputs=x)

input_params = Input(shape=(11,))
y = Dense(128, activation='relu')(input_params)
model_params = Model(inputs=input_params, outputs=y)

combined = Concatenate()([model_flux.output, model_params.output])
z = Dense(128, activation='relu')(combined)
final_output = Dense(1, activation='sigmoid')(z)

model = Model(inputs=[model_flux.input, model_params.input], outputs=final_output)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
# history = model.fit([X_train_flux, X_train_params], y_train, epochs=15, batch_size=32, validation_split=0.2)


2024-03-25 20:00:17.523520: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2024-03-25 20:00:17.523698: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2024-03-25 20:00:17.523708: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2024-03-25 20:00:17.523746: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2024-03-25 20:00:17.523771: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [7]:
model.load_weights('model_weights_88_accuracy.weights.h5')

/Users/nasvinnabeel/anaconda3/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:396: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 22 variables. 
  trackable.load_own_variables(weights_store.get(inner_path))


In [8]:
model.evaluate([X_test_flux,X_test_params],y_test)

2024-03-25 20:00:18.038350: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8835 - loss: 0.4699


[0.40077731013298035, 0.8888888955116272]

In [51]:
# model.save_weights('model_weights_88_accuracy.weights.h5')